# NLP + Transformers


---
## 1) Visualización de la Configuración de BERT 👀

In [1]:
#!pip install transformers

In [2]:
from transformers import BertConfig, BertModel

C:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Cargar la configuración del modelo BERT en español

In [3]:
config = BertConfig.from_pretrained("dccuchile/bert-base-spanish-wwm-cased")
print("Número de capas de BERT:", config.num_hidden_layers)

Número de capas de BERT: 12


In [4]:
print("hidden_size (d_model):", config.hidden_size)
print("Cabezas de atención:", config.num_attention_heads)
print("Dimensión por cabeza (d_k):", config.hidden_size // config.num_attention_heads)
print("intermediate_size (FFN):", config.intermediate_size)
print("vocab_size:", config.vocab_size)

hidden_size (d_model): 768
Cabezas de atención: 12
Dimensión por cabeza (d_k): 64
intermediate_size (FFN): 3072
vocab_size: 31002


Cargar el modelo para visualizar su arquitectura

In [5]:
model = BertModel.from_pretrained("dccuchile/bert-base-spanish-wwm-cased", output_hidden_states=True)
print(model)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 37810.73it/s]


[transformers] BertModel LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(31002, 768, padding_idx=1)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
            (dropout): Dropout(p=

In [6]:
n_params = sum(p.numel() for p in model.parameters())
print(f"Parámetros totales: {n_params:,}")

Parámetros totales: 109,850,880


### 📝 Análisis de la Sección 1: Configuración y arquitectura de BERT (BETO)

**1. Los tres módulos principales de `print(model)`**

| Módulo | Qué contiene | Función |
|---|---|---|
| `embeddings` (`BertEmbeddings`) | `word_embeddings` Embedding(31002, 768), `position_embeddings` Embedding(512, 768), `token_type_embeddings` Embedding(2, 768), `LayerNorm` y `dropout` | Convierte cada `input_id` en un vector de 768 dimensiones. Suma tres vectores: el del token (significado), el de su posición (orden) y el del segmento (oración A o B). Luego aplica LayerNorm y dropout. Es la entrada del primer bloque. |
| `encoder` (`BertEncoder`) | `ModuleList` de **12 × `BertLayer`** (índices `0-11`) | Es el núcleo del modelo. Cada `BertLayer` aplica autoatención multi‑cabeza y una red Feed‑Forward, ambas con conexión residual y LayerNorm. Las 12 capas se apilan: la salida de una es la entrada de la siguiente. |
| `pooler` (`BertPooler`) | `dense` Linear(768, 768) + activación `Tanh` | Toma solo el vector del token `[CLS]` (posición 0) y lo proyecta a un resumen de la oración. Se usa en tareas de clasificación a nivel de secuencia. |

> ⚠️ **Observación del reporte de carga:** en la salida de `from_pretrained` aparecen `pooler.dense.weight` y `pooler.dense.bias` como **MISSING**. El checkpoint de BETO se entrenó solo con MLM, así que el pooler se inicializó aleatoriamente. Por eso `pooler_output` no es una representación fiable sin *fine‑tuning*. Las claves `cls.predictions.*` aparecen como UNEXPECTED porque pertenecen a la cabeza de lenguaje enmascarado, que `BertModel` no incluye. Ambos avisos son normales y no afectan a los hidden states del encoder.

**2. Subcapas dentro de una `BertLayer`**

*Subcapas del mecanismo de atención* (`attention.self` = `BertSelfAttention`):
- `query` = `Linear(768, 768)`: proyecta cada token a su vector **Q** (qué información busca).
- `key` = `Linear(768, 768)`: proyecta cada token a su vector **K** (qué información ofrece).
- `value` = `Linear(768, 768)`: proyecta cada token a su vector **V** (el contenido que se mezcla).
- `dropout` (p = 0.1): se aplica sobre los pesos de atención (solo en entrenamiento).

Cada proyección de 768 dimensiones se divide en **12 cabezas de 64 dimensiones** (`d_k = 768 / 12`).

*Subcapas de conexión residual y normalización:*
- `attention.output` (`BertSelfOutput`) contiene `dense` Linear(768, 768) (la proyección de salida **W_O** que recombina las cabezas), `dropout` y `LayerNorm((768,), eps=1e-12)`. En su `forward` se calcula `LayerNorm(dropout(dense(contexto)) + entrada)`: la suma con la entrada es la **conexión residual**. Se ve en el código pero no como módulo impreso, porque es una operación y no una capa con parámetros.
- `output` (`BertOutput`) repite el mismo patrón tras el Feed‑Forward: `dense` Linear(3072 → 768), `dropout` y `LayerNorm`, con residual.
- `intermediate` (`BertIntermediate`) es la expansión Linear(768 → 3072) con activación GELU. Es la primera mitad de la red Feed‑Forward.

Así, cada capa sigue el esquema **Post‑LN**: `x → Atención → Add & LayerNorm → Feed‑Forward → Add & LayerNorm`.

**3. Dimensión oculta (`hidden_size` o d_model)**

`config.hidden_size = 768`, y es la misma en todo el modelo. Las capas lineales de Q, K, V y de la proyección de salida son 768 → 768. El Feed‑Forward expande a `intermediate_size = 3072` (4 × 768) y vuelve a 768. Con 12 cabezas, `d_k = d_v = 64`. El modelo tiene **109,850,880 parámetros**; la mayor parte está en las 12 capas del encoder.


## 2) Visualización de las Capas y Hidden States 🔍

In [7]:
from transformers import BertTokenizer
import torch

Cargar tokenizer y modelo BERT en español

In [8]:
tokenizer = BertTokenizer.from_pretrained("dccuchile/bert-base-spanish-wwm-cased")
model = BertModel.from_pretrained("dccuchile/bert-base-spanish-wwm-cased", output_hidden_states=True)

C:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 11876.59it/s]


[transformers] BertModel LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Texto de ejemplo

In [9]:
text = "El producto de Samsung Galaxy S21 llegó el 12 de marzo y superó mis expectativas."
inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
outputs = model(**inputs)

In [10]:
print("Claves del diccionario inputs:", list(inputs.keys()))
print("input_ids      :", inputs['input_ids'])
print("token_type_ids:", inputs['token_type_ids'])
print("attention_mask:", inputs['attention_mask'])
print("\nTokens (WordPiece):")
print(tokenizer.convert_ids_to_tokens(inputs['input_ids'][0]))
print("Longitud de la secuencia:", inputs['input_ids'].shape[1])

Claves del diccionario inputs: ['input_ids', 'token_type_ids', 'attention_mask']
input_ids      : tensor([[    4,  1162,  4196,  1008,  4262,  8643, 16685, 21291, 24010,  1057,
          8926,  4444,  1040,  1797,  1008,  3008,  1042, 27994,  1354, 12956,
          1009,     5]])
token_type_ids: tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])
attention_mask: tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])

Tokens (WordPiece):
['[CLS]', 'El', 'producto', 'de', 'Sam', '##su', '##ng', 'Gala', '##xy', 'S', '##21', 'llegó', 'el', '12', 'de', 'marzo', 'y', 'superó', 'mis', 'expectativas', '.', '[SEP]']
Longitud de la secuencia: 22


In [11]:
print(outputs)

BaseModelOutputWithPoolingAndCrossAttentions(last_hidden_state=tensor([[[-0.3688, -0.2218, -0.6865,  ..., -0.4430,  0.2883,  0.1794],
         [-0.2616, -0.0662, -0.6243,  ..., -0.5904,  0.6507,  0.2768],
         [-0.2415, -0.2261, -0.6785,  ..., -0.1096,  0.0910,  0.5395],
         ...,
         [-1.4393, -0.1986, -0.5770,  ..., -0.5261,  0.0716,  1.0376],
         [-0.7026,  0.3509, -0.4953,  ..., -0.3322,  0.3734,  0.1284],
         [-1.2273, -0.4597, -1.0296,  ..., -0.6864,  0.9237, -0.5841]]],
       grad_fn=<NativeLayerNormBackward0>), pooler_output=tensor([[ 5.0187e-01,  2.5280e-01, -9.6470e-03,  2.1693e-01,  5.2045e-01,
         -2.9275e-01,  7.4708e-01,  2.5321e-01, -6.8669e-02,  3.0614e-01,
         -1.2357e-01,  7.1154e-02, -1.3610e-01, -2.4062e-01, -7.2927e-01,
         -2.6269e-01,  1.7740e-01,  2.7583e-01,  5.2125e-01, -8.3430e-02,
          6.7692e-01,  1.4674e-02,  5.6666e-01,  6.9600e-01, -6.2046e-02,
         -5.3807e-01, -3.0352e-01,  3.5055e-01,  2.4817e-01,  3.876

Extraer los hidden states


In [12]:
hidden_states = outputs.hidden_states
print("Número de capas (incluyendo la capa de embedding):", len(hidden_states))

Número de capas (incluyendo la capa de embedding): 13


In [13]:
print("Forma de hidden_states[0] (embeddings):", hidden_states[0].shape)
print("Forma de hidden_states[12] (última capa):", hidden_states[12].shape)
print("\nForma de cada hidden state:")
for i, h in enumerate(hidden_states):
    print(f"  hidden_states[{i:2d}] -> {tuple(h.shape)}")
print("\n¿La última capa coincide con last_hidden_state?", torch.equal(hidden_states[-1], outputs.last_hidden_state))
print("Forma de pooler_output:", outputs.pooler_output.shape)

Forma de hidden_states[0] (embeddings): torch.Size([1, 22, 768])
Forma de hidden_states[12] (última capa): torch.Size([1, 22, 768])

Forma de cada hidden state:
  hidden_states[ 0] -> (1, 22, 768)
  hidden_states[ 1] -> (1, 22, 768)
  hidden_states[ 2] -> (1, 22, 768)
  hidden_states[ 3] -> (1, 22, 768)
  hidden_states[ 4] -> (1, 22, 768)
  hidden_states[ 5] -> (1, 22, 768)
  hidden_states[ 6] -> (1, 22, 768)
  hidden_states[ 7] -> (1, 22, 768)
  hidden_states[ 8] -> (1, 22, 768)
  hidden_states[ 9] -> (1, 22, 768)
  hidden_states[10] -> (1, 22, 768)
  hidden_states[11] -> (1, 22, 768)
  hidden_states[12] -> (1, 22, 768)

¿La última capa coincide con last_hidden_state? True
Forma de pooler_output: torch.Size([1, 768])


Por ejemplo, si BERT base tiene 12 capas, se imprimirá 13 (1 de embedding + 12 codificadores)

### 📝 Análisis de la Sección 2: Tokenización y Hidden States

**1. El diccionario `inputs`**

La oración quedó dividida en **22 tokens** (WordPiece, vocabulario de 31,002 entradas, *cased*):
`[CLS] El producto de Sam ##su ##ng Gala ##xy S ##21 llegó el 12 de marzo y superó mis expectativas . [SEP]`

- **`input_ids`** es la secuencia de enteros que identifica cada token en el vocabulario de BETO. Incluye los tokens especiales `[CLS]` (id 4, al inicio, su estado final resume la oración) y `[SEP]` (id 5, al final). Las palabras fuera de vocabulario se parten en subpalabras: *Samsung* → `Sam`, `##su`, `##ng`; *Galaxy* → `Gala`, `##xy`; *S21* → `S`, `##21`. El prefijo `##` indica que la pieza continúa la palabra anterior. Estos ids son los índices que consulta `word_embeddings`.
- **`token_type_ids`** es el *segment id*: indica a qué oración pertenece cada token (0 = primera oración, 1 = segunda). Sirve para tareas con dos textos, como inferencia entre oraciones o pregunta‑respuesta, y es el índice de `token_type_embeddings` (2 × 768). Aquí todos son **0** porque hay una sola oración.
- **`attention_mask`** marca con **1** los tokens reales y con **0** los de relleno (*padding*). Antes del softmax, las posiciones con 0 reciben un puntaje de −∞ y su peso de atención queda en 0. Aquí **todos son 1** porque procesamos una sola oración (batch de 1): no hay padding, todos los tokens son reales y todos deben participar en la atención.

**2. Por qué `len(hidden_states)` es 13 y no 12**

`config.num_hidden_layers = 12` cuenta solo los bloques del encoder. La tupla `hidden_states` tiene **13 tensores**:

| Posición | Contenido |
|---|---|
| `hidden_states[0]` | **La salida del módulo `embeddings`**: embedding de token + posición + segmento, con LayerNorm y dropout. Es el tensor que entra al primer `BertLayer`, **antes de cualquier capa de atención**. Cada token aún no tiene contexto: su vector depende solo de su identidad y su posición. |
| `hidden_states[1]` … `[12]` | La salida de cada una de las 12 capas del encoder. Cada una mezcla más contexto de la oración. |

Por tanto, 13 = 1 (embeddings) + 12 (encoder). En el notebook, `hidden_states[12]` es idéntico a `outputs.last_hidden_state` (`torch.equal` devolvió `True`).

**3. Forma de `hidden_states[0]`: `torch.Size([1, 22, 768])`**

| Dimensión | Valor | Significado |
|---|---|---|
| `batch_size` | **1** | Número de oraciones procesadas a la vez. Aquí es una. |
| `sequence_length` | **22** | Número de tokens de la secuencia, incluidos `[CLS]` y `[SEP]`. Depende del texto y de la tokenización. |
| `hidden_size` | **768** | Tamaño del vector que representa a cada token (d_model). Es constante en todo el modelo. |

Como se imprimió la forma de las 13 salidas, se ve que todas son `(1, 22, 768)`. Cada capa conserva la forma del tensor, y esto es lo que hace posible la conexión residual (`x + capa(x)`). Lo único que cambia entre capas es el *contenido* de los 768 números, que pasa de una representación léxica a una contextual.


## 3) Diagrama Conceptual y Pseudocódigo 🙌

Pseudocódigo del mecanismo de Self-Attention

**Entradas y parámetros**

```
ENTRADA : X ∈ ℝ^(n × d_model)        # hidden state de la capa anterior (aquí n = 22, d_model = 768)
          M ∈ ℝ^(n × n)              # máscara: 0 en tokens reales, −∞ en padding (derivada de attention_mask)
PARAMS  : W_Q, W_K, W_V ∈ ℝ^(768 × 768)  con sus sesgos b_Q, b_K, b_V   # capas query, key, value
          W_O ∈ ℝ^(768 × 768) con sesgo b_O                              # attention.output.dense
HIPERP. : h = 12 cabezas ;  d_k = d_v = d_model / h = 64
```

**Pseudocódigo paso a paso**

```
FUNCIÓN SelfAttention(X, M):

  # ── 1. Proyección lineal de los tokens ───────────────────────────────
  Q ← X · W_Q + b_Q          # [n, 768]   "qué busca cada token"
  K ← X · W_K + b_K          # [n, 768]   "qué ofrece cada token"
  V ← X · W_V + b_V          # [n, 768]   "qué contenido aporta cada token"

  # ── 2. División en h cabezas (Multi-Head) ────────────────────────────
  PARA i = 1 … h:            # las 12 cabezas son independientes
      Q_i, K_i, V_i ← i-ésimo bloque de 64 columnas de Q, K, V      # [n, 64] cada uno

      # ── 3. Puntaje de similitud entre todos los pares de tokens ──────
      S_i ← Q_i · K_iᵀ                    # [n, n]  S_i[a,b] = qué tanto "mira" a al token b

      # ── 4. Escalado ─────────────────────────────────────────────────
      S_i ← S_i / √d_k                    # √64 = 8: evita que el softmax se sature

      # ── 5. Enmascarado ──────────────────────────────────────────────
      S_i ← S_i + M                       # los tokens de padding quedan con peso 0

      # ── 6. Normalización con softmax (por fila) ──────────────────────
      A_i ← softmax(S_i, eje = columnas)  # [n, n]  cada fila suma 1 = pesos de atención

      # ── 7. Suma ponderada de los valores ─────────────────────────────
      Z_i ← A_i · V_i                     # [n, 64]  cada token = mezcla de los V según A_i

  # ── 8. Concatenación de cabezas y proyección de salida ───────────────
  Z ← concat(Z_1, …, Z_h)                 # [n, 768]
  Z ← Z · W_O + b_O                       # [n, 768]  recombina la información de las cabezas

  # ── 9. Residual + LayerNorm (BertSelfOutput) ─────────────────────────
  RETORNAR LayerNorm(X + Z)               # [n, 768]  misma forma que la entrada
```

Esto equivale a la fórmula `Attention(Q, K, V) = softmax(Q·Kᵀ / √d_k) · V`, aplicada en paralelo a cada cabeza.

**Diagrama conceptual (flujo de tensores en una cabeza)**

```
 X [n,768] ──► W_Q ──► Q [n,64] ──┐
     │                             ├─► Q·Kᵀ [n,n] ─► ÷√d_k ─► +M ─► softmax ─► A [n,n] ──┐
     ├──────► W_K ──► K [n,64] ──┘                                                      ├─► A·V [n,64]
     └──────► W_V ──► V [n,64] ─────────────────────────────────────────────────────────┘
                                                      │ (12 cabezas) concat [n,768] ─► W_O ─► + X ─► LayerNorm
```

**Ejemplo con nuestra oración:** para el token `llegó` (posición 11), la fila `A_i[11, :]` es una distribución de 22 pesos que suma 1. Indica cuánto toma cada token de la oración (`producto`, `marzo`, `expectativas`, …) para construir la nueva representación de `llegó` en esa cabeza. Cada una de las 12 cabezas aprende un patrón distinto.


Verificación numérica del pseudocódigo con los pesos reales de BETO (capa 0):

In [14]:
import math

# Verificación numérica: reproducimos a mano la capa BertSelfAttention de la capa 0
# usando los pesos reales de Q, K, V y comparamos contra la salida del modelo.
model.eval()
attn = model.encoder.layer[0].attention.self
x = hidden_states[0]                      # [1, n, 768] entrada de la capa 0
H, d_model = config.num_attention_heads, config.hidden_size
d_k = d_model // H

with torch.no_grad():
    Q = attn.query(x); K = attn.key(x); V = attn.value(x)          # proyecciones lineales
    split = lambda t: t.view(1, -1, H, d_k).transpose(1, 2)        # [1, H, n, d_k]
    Qh, Kh, Vh = split(Q), split(K), split(V)
    scores  = Qh @ Kh.transpose(-2, -1) / math.sqrt(d_k)           # QK^T / sqrt(d_k)
    weights = torch.softmax(scores, dim=-1)                        # softmax por fila
    context = (weights @ Vh).transpose(1, 2).reshape(1, -1, d_model)  # suma ponderada y concatenación
    ref = attn(x)
    ref = ref[0] if isinstance(ref, tuple) else ref

print("scores :", tuple(scores.shape), "  (batch, cabezas, n, n)")
print("weights:", tuple(weights.shape), " suma de cada fila =", weights[0,0,0].sum().item())
print("context:", tuple(context.shape))
print("Diferencia máxima vs BertSelfAttention:", (context - ref).abs().max().item())

scores :

 (1, 12, 22, 22)   (batch, cabezas, n, n)
weights: (1, 12, 22, 22)  suma de cada fila = 1.0
context: (1, 22, 768)
Diferencia máxima vs BertSelfAttention: 4.76837158203125e-07


### 📝 Análisis de la Sección 3: Pseudocódigo y relación con `BertSelfAttention`

**La celda anterior valida el pseudocódigo con los pesos reales.** Se tomó `model.encoder.layer[0].attention.self` y `hidden_states[0]`. Con los pesos `query`, `key` y `value` del propio modelo se calcularon a mano los pasos 1 a 8. El resultado se comparó con la salida de `BertSelfAttention`: la diferencia máxima fue **4.8 × 10⁻⁷**, es decir, error numérico de punto flotante. Además `scores` y `weights` tienen forma `(1, 12, 22, 22)` (una matriz 22 × 22 por cabeza) y cada fila de `weights` suma 1.0, tal como indica el paso 6.

**1. Resumen operación por operación**

| Paso | Operación | Forma resultante |
|---|---|---|
| 1 | Proyección lineal `X·W_Q, X·W_K, X·W_V` | 3 × `[22, 768]` |
| 2 | División en 12 cabezas | 12 × `[22, 64]` por cada Q, K, V |
| 3 | `Q·Kᵀ`: similitud entre todos los pares de tokens | `[22, 22]` por cabeza |
| 4 | División por `√d_k = 8`: mantiene la varianza de los puntajes estable | `[22, 22]` |
| 5 | Suma de la máscara: padding recibe −∞ (aquí no hay) | `[22, 22]` |
| 6 | `softmax` por filas: convierte los puntajes en pesos que suman 1 | `[22, 22]` |
| 7 | `A·V`: suma ponderada de los valores | `[22, 64]` por cabeza |
| 8 | Concatenar las 12 cabezas y proyectar con `W_O` | `[22, 768]` |
| 9 | Residual y LayerNorm | `[22, 768]` |

**2. Por qué el diagrama y el pseudocódigo van al cierre del laboratorio**

Van al final porque son la **síntesis** de lo observado antes. El orden del laboratorio es de lo concreto a lo abstracto:
1. En la **Sección 1** se vio *qué módulos existen* (`embeddings`, `encoder`, `pooler`) y que `BertSelfAttention` contiene `query`, `key` y `value`, pero solo como capas lineales impresas, sin su lógica.
2. En la **Sección 2** se vio *qué viaja por ellos*: tensores `[1, 22, 768]` que conservan su forma a lo largo de las 13 salidas.
3. En la **Sección 3** se explica *qué cálculo ocurre dentro* de cada `BertSelfAttention`. Sin los dos pasos anteriores, el pseudocódigo sería abstracto.

**Relación con lo impreso en la Sección 1:**

| Elemento del pseudocódigo | Módulo en `print(model)` |
|---|---|
| Paso 1: `W_Q`, `W_K`, `W_V` | `query`, `key`, `value` = `Linear(768, 768)` dentro de `BertSelfAttention` |
| Pasos 2 a 7 (cabezas, `QKᵀ/√d_k`, softmax, `A·V`) | No tienen parámetros, por eso no aparecen en `print(model)`; están en el `forward` de `BertSelfAttention`. El `dropout(p=0.1)` actúa sobre `A` en entrenamiento. |
| Paso 8: `W_O` | `attention.output.dense` (`BertSelfOutput`) |
| Paso 9: residual y LayerNorm | `attention.output.LayerNorm` (`BertSelfOutput`) |

Las 12 capas del encoder repiten este bloque. La salida de cada una es el siguiente `hidden_states[k]`. Por eso las 12 capas se pueden apilar sin cambiar la forma `[1, 22, 768]`, y por eso `hidden_states[0]` (entrada de la capa 1) existe aparte de las 12 salidas.
